# Cav1.2 G402S: backbone geometry and mutant-site proximity

WT and G402S are compared using matched Cα–Cα coordinates in vanilla data. The corrected WT masked candidate is available, but a corrected G402S masked cohort is not; legacy masked data are excluded and the masked mutant comparison is deferred. Mutation-neighborhood Cα panels and a dedicated position-402 panel describe backbone geometry.

Legacy `shortest_*` columns are retained only as historical field names; the current mutant analysis is vanilla-only until corrected G402S masked data exist. WT experimental glycine references occupy a separate panel because they are all-heavy-atom minima and are not an atom-matched mutant comparison. Shorter minimum distances can arise from the added side chain and its rotamers without backbone movement.

## Distance definitions and interpretation

- **Matched backbone metric:** `CA_GLY402_CA-PARTNER_CA` in WT and `CA_SER402_CA-PARTNER_CA` in G402S. Both endpoints are Cα atoms in both sequences. Missing or nonfinite values stop the new comparison; no shortest-distance fallback is allowed.
- **WT experimental minima:** the minimum Euclidean distance over all non-hydrogen atoms of both residues. Glycine contributes its backbone N, Cα, C and O atoms, not a side chain. The exported reference table identifies the closest atom pair and separately reports Cα–Cα distances.
- **Historical model minima:** the notebooks consume precomputed `shortest_*` CSV columns. Their original PDB-generator atom selection has not been verified. A separate local CIF tool uses glycine Cα and selected terminal atoms, so equivalence to the experimental all-heavy definition must not be assumed.
- **Interpretation:** matched Cα changes measure backbone geometry. Minimum-distance changes combine backbone movement, side-chain reach and rotamers; WT–mutant values additionally change atom composition. Neither proximity nor a 2–4 Å threshold alone establishes hydrogen bonding or a salt bridge. Values below 2 Å are overlap flags.

All snapshot panels use the explicitly selected cohort below and are descriptive, not tests treating snapshots as independent replicates. WT and G402S panels use matched Cα endpoints; historical shortest-distance minima are excluded because their atom-selection rule is not recoverable.

In [1]:
from pathlib import Path
import importlib
import sys
repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
import shared.mutation_site_analysis as mutation_site_analysis
importlib.reload(mutation_site_analysis)  # Pick up helper changes in an already-running kernel.
load_ensemble = mutation_site_analysis.load_ensemble
nearby_distance_columns = mutation_site_analysis.nearby_distance_columns
rank_nearby_shifts = mutation_site_analysis.rank_nearby_shifts
top_aliases = mutation_site_analysis.top_aliases
plot_nearby_overlay = mutation_site_analysis.plot_nearby_overlay
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv, cav12_active_masked_distance_path
from shared.plotting import CAV12_PALETTE, plot_top_shifts
G402S_EXPERIMENTAL_DISTANCES = {
    '8WE6 | chain A': {
        'CA_LEU401_CA-ILE1186_CA': [10.070], 'CA_LEU401_CA-VAL753_CA': [11.718], 'GLY406-MET1524': [7.404],
        'LEU401-ALA1521': [9.341], 'LEU401-ASN1526': [9.824], 'LEU401-ASP1525': [10.976], 'LEU401-ASP1528': [11.996], 'LEU401-ASP1533': [14.828],
        'LEU401-MET1524': [8.529], 'LEU401-TYR1529': [10.879], 'LEU401-VAL1522': [8.829], 'LEU404-MET1524': [9.680],
        'SER405-ASP1525': [9.391], 'SER405-ASP1533': [11.521], 'SER405-MET1524': [6.077], 'SER405-TYR1529': [9.355],
        'VAL400-ASP1528': [13.981], 'VAL400-MET1524': [11.185], 'VAL403-ASP1528': [10.703], 'VAL403-MET1524': [9.090],
    },
    '8HLP | chain A': {
        'CA_LEU401_CA-ILE1186_CA': [9.986], 'CA_LEU401_CA-VAL753_CA': [11.751], 'GLY406-MET1524': [8.961],
        'LEU401-ALA1521': [9.471], 'LEU401-ASN1526': [9.889], 'LEU401-ASP1525': [11.505], 'LEU401-ASP1528': [11.921], 'LEU401-ASP1533': [14.849],
        'LEU401-MET1524': [9.147], 'LEU401-TYR1529': [10.943], 'LEU401-VAL1522': [8.975], 'LEU404-MET1524': [11.400],
        'SER405-ASP1525': [9.994], 'SER405-ASP1533': [11.415], 'SER405-MET1524': [7.551], 'SER405-TYR1529': [9.544],
        'VAL400-ASP1528': [14.009], 'VAL400-MET1524': [11.875], 'VAL403-ASP1528': [10.726], 'VAL403-MET1524': [9.957],
    },
    '8FD7 | chain K': {
        'CA_LEU401_CA-ILE1186_CA': [13.653], 'CA_LEU401_CA-VAL753_CA': [15.182], 'GLY406-MET1524': [13.416],
        'LEU401-ALA1521': [7.614], 'LEU401-ASN1526': [3.233], 'LEU401-ASP1525': [7.669], 'LEU401-ASP1528': [8.778], 'LEU401-ASP1533': [11.673],
        'LEU401-MET1524': [6.187], 'LEU401-TYR1529': [8.138], 'LEU401-VAL1522': [5.311], 'LEU404-MET1524': [8.234],
        'SER405-ASP1525': [11.258], 'SER405-ASP1533': [7.884], 'SER405-MET1524': [10.802], 'SER405-TYR1529': [6.804],
        'VAL400-ASP1528': [13.343], 'VAL400-MET1524': [11.054], 'VAL403-ASP1528': [13.659], 'VAL403-MET1524': [13.091],
    },
}
G402S_EXPERIMENTAL_DISTANCES

from shared.plotting import apply_kv21_style
apply_kv21_style()


In [2]:
from shared.presentation_export import install_notebook_figure_export
figure_dir = repo_root / 'cav12' / 'dataDistances' / 'analysis' / 'figures'
table_dir = repo_root / 'cav12' / 'dataDistances' / 'analysis' / 'tables'
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)
# Export every displayed panel locally, including the neighborhood comparisons.
install_notebook_figure_export(figure_dir, 'Cav12_G402S_mutationSite_analysis')


PosixPath('/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/analysis/figures/Cav12_G402S_mutationSite_analysis')

In [3]:
DATASET_SELECTION = 'all_ok_3'  # Corrected 3 Å convergence subset stored in cav12/dataDistances; switch to 'all' to inspect unfiltered rows.
data = repo_root / 'cav12' / 'dataDistances'
DATASET_PATHS = {
    'WT vanilla': distance_csv_options(repo_root, data / '26-02-10_Cav12_wt_vanillaAF2_distances_all.csv', 'Cav12', 'WT', 'vanilla'),
    'WT masked': distance_csv_options(repo_root, cav12_active_masked_distance_path(repo_root, 'WT'), 'Cav12', 'WT', 'masked'),
    'G402S vanilla': distance_csv_options(repo_root, data / '26-02-10_Cav12_g402s_vanillaAF2_distances_all.csv', 'Cav12', 'G402S', 'vanilla'),
    'G402S masked': distance_csv_options(repo_root, cav12_active_masked_distance_path(repo_root, 'G402S'), 'Cav12', 'G402S', 'masked'),
}
DATASET_PATHS
wt_van = load_selected_distance_csv('WT vanilla', DATASET_PATHS['WT vanilla'], DATASET_SELECTION, fallback_to_all=False)
wt_mask = load_selected_distance_csv('WT masked', DATASET_PATHS['WT masked'], DATASET_SELECTION, fallback_to_all=False)
mut_van = load_selected_distance_csv('G402S vanilla', DATASET_PATHS['G402S vanilla'], DATASET_SELECTION, fallback_to_all=False)
mut_mask = load_selected_distance_csv('G402S masked', DATASET_PATHS['G402S masked'], DATASET_SELECTION, fallback_to_all=False)
structures = {'8WE6 | chain A': (repo_root / 'cav12' / 'experimental' / '8WE6.pdb', 'A'), '8HLP | chain A': (repo_root / 'cav12' / 'experimental' / '8HLP.pdb', 'A'), '8FD7 | chain K': (repo_root / 'cav12' / 'experimental' / '8FD7.pdb', 'K')}
experimental_colors = None  # shared PDB-specific colors and markers


/Users/ahernandezgonzalez/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WT vanilla: requested=all_ok_3; actual=all_ok_3; path=/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/26-02-10_Cav12_wt_vanillaAF2_distances_all_ok_rmsd_3A.csv
Merged companion distance table: cav12/dataDistances/26-11-10_Cav12_wt_masked_v3_50pct_AF2_distances_extra_all_ok_rmsd_3A.csv
WT masked: requested=all_ok_3; actual=all_ok_3; path=cav12/dataDistances/26-11-10_Cav12_wt_masked_v3_50pct_AF2_distances_all_ok_rmsd_3A.csv


G402S vanilla: requested=all_ok_3; actual=all_ok_3; path=/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/26-02-10_Cav12_g402s_vanillaAF2_distances_all_ok_rmsd_3A.csv
Merged companion distance table: cav12/dataDistances/26-11-10_Cav12_g402s_masked_v3_50pct_AF2_distances_extra_all_ok_rmsd_3A.csv
G402S masked: requested=all_ok_3; actual=all_ok_3; path=cav12/dataDistances/26-11-10_Cav12_g402s_masked_v3_50pct_AF2_distances_all_ok_rmsd_3A.csv


## Shared mutation-neighborhood Cα coordinates

These ranked comparisons include only Cα–Cα columns present in both sequences. They describe surrounding backbone geometry; the explicit mutation-site comparison follows below. Historical shortest-distance columns are excluded from this backbone ranking.

In [4]:
for protocol, wt, mutant, colors in [('vanilla', wt_van, mut_van, (CAV12_PALETTE['WT_VAN'], CAV12_PALETTE['G402S_VAN'])), ('masked', wt_mask, mut_mask, (CAV12_PALETTE['WT_HM'], CAV12_PALETTE['G402S_HM']))]:
    shifts = rank_nearby_shifts(wt, mutant, [c for c in nearby_distance_columns(wt, 402) if c.startswith("CA_")])
    display(shifts.head(20))
    if shifts.empty:
        print(f'No complete WT–G402S paired distances are available for {protocol}; skipping the ranked-shift overlay.')
        continue
    plot_top_shifts(shifts, f'Largest Cav1.2 mutation-neighborhood Cα shifts: G402S {protocol}')
    aliases = top_aliases(shifts)
    experimentals = {structure: {alias: values for alias, values in distances.items() if alias in aliases} for structure, distances in G402S_EXPERIMENTAL_DISTANCES.items()}
    plot_nearby_overlay(wt, mutant, aliases, 'Cav1.2', 'G402S', protocol, colors, experimentals, experimental_colors)



,distance,wt_median_A,mutant_median_A,median_shift_A,abs_shift_A
36,CA_LEU401_CA-THR1531_CA,12.80,13.18,0.38,0.38
35,CA_LEU401_CA-LEU1530_CA,14.51,14.87,0.36,0.36
104,CA_GLY406_CA-THR1531_CA,7.88,8.22,0.34,0.34
87,CA_SER405_CA-THR1531_CA,11.09,11.43,0.34,0.34
86,CA_SER405_CA-LEU1530_CA,13.49,13.80,0.31,0.31
33,CA_LEU401_CA-ASP1528_CA,10.63,10.94,0.31,0.31
34,CA_LEU401_CA-TYR1529_CA,14.22,14.53,0.31,0.31
93,CA_GLY406_CA-VAL1520_CA,13.41,13.11,-0.30,0.30
30,CA_LEU401_CA-ASP1525_CA,11.45,11.74,0.29,0.29
106,CA_GLY406_CA-ASP1533_CA,10.19,10.48,0.29,0.29


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


,distance,wt_median_A,mutant_median_A,median_shift_A,abs_shift_A
1,CA_LEU401_CA-VAL753_CA,11.38,12.76,1.38,1.38
88,CA_SER405_CA-ARG1532_CA,11.28,12.60,1.32,1.32
33,CA_LEU401_CA-ASP1528_CA,11.10,12.42,1.32,1.32
84,CA_SER405_CA-ASP1528_CA,8.74,10.00,1.26,1.26
85,CA_SER405_CA-TYR1529_CA,12.55,13.78,1.23,1.23
34,CA_LEU401_CA-TYR1529_CA,14.58,15.76,1.18,1.18
105,CA_GLY406_CA-ARG1532_CA,7.68,8.85,1.17,1.17
37,CA_LEU401_CA-ARG1532_CA,15.04,16.16,1.12,1.12
30,CA_LEU401_CA-ASP1525_CA,11.72,12.84,1.12,1.12
106,CA_GLY406_CA-ASP1533_CA,10.23,11.35,1.12,1.12


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


## Matched position-402 Cα comparison

WT glycine and mutant SER402 contribute Cα only in this panel; experimental markers also use Cα–Cα distances. The exported table records each source column, model identifier, and selected cohort.

In [5]:
from shared.cav12_glycine_analysis import plot_matched_ca
from shared.mutation_contact_analysis import cav12_experimental_shortest_distances
import matplotlib.pyplot as plt

partners = ['ILE1523', 'VAL1520', 'MET1524', 'ASP1528']
experimental_geometry = cav12_experimental_shortest_distances(repo_root, 402, partners)
comparisons = {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)}
matched_ca, matched_ca_figure = plot_matched_ca(
    comparisons, 402, 'SER402', 'G402S', partners, experimental_geometry
)
matched_ca.to_csv(table_dir / 'g402s_matched_ca_plot_data.csv', index=False)
display(matched_ca.groupby(['Protocol', 'Sequence', 'Partner'])['Cα distance (Å)'].agg(['count', 'median']))
for extension in ('png', 'pdf'):
    matched_ca_figure.savefig(figure_dir / f'g402s_matched_ca_comparison.{extension}', dpi=300, bbox_inches='tight')
plt.show()


count  median
Protocol Sequence Partner               
masked   G402S    ASP1528   4993    8.68
                  ILE1523   4993    7.24
                  MET1524   4993    5.48
                  VAL1520   4993    7.07
         WT       ASP1528   4992    7.39
                  ILE1523   4992    7.24
                  MET1524   4992    5.02
                  VAL1520   4992    7.74
vanilla  G402S    ASP1528   5000    7.19
                  ILE1523   5000    7.64
                  MET1524   5000    4.96
                  VAL1520   5000    7.99
         WT       ASP1528   5000    6.85
                  ILE1523   5000    7.65
                  MET1524   5000    4.89
                  VAL1520   5000    8.23

/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


## Paired-seed revision: sequence effect versus masking effect

This panel reads the current `analysis/statistics_revision/paired_seed_v2` matched-Cα source tables. It shows all 17 position-402-to-partner pairs. Vanilla WT–G402S is the primary mutation sequence contrast; G402S vanilla–masked is a separate masking effect. No nearest-partner category is used.


In [6]:
from shared.cav12_glycine_analysis import plot_g402s_paired_revision
paired_effects, paired_partners, paired_figure = plot_g402s_paired_revision(repo_root)
paired_partners.to_csv(table_dir / 'g402s_all_partner_categories.csv', index=False)
display(paired_effects[['contrast', 'estimate_A', 'estimate_B', 'estimate_B_minus_A', 'CI_low_B_minus_A', 'CI_high_B_minus_A']])
display(paired_partners)
for extension in ('png', 'pdf'):
    paired_figure.savefig(figure_dir / f'g402s_paired_seed_backbone_revision.{extension}', dpi=300, bbox_inches='tight')
plt.show()


,contrast,estimate_A,estimate_B,estimate_B_minus_A,CI_low_B_minus_A,CI_high_B_minus_A
0,vanilla G402S - vanilla WT,11.84355,11.69764,-0.14591,-0.158481,-0.133960
6,G402S masked - G402S vanilla,11.69764,10.57808,-1.11956,-1.135926,-1.103107
9,vanilla G402S - vanilla WT,8.30457,8.12311,-0.18146,-0.192333,-0.169899
15,G402S masked - G402S vanilla,8.12311,7.09404,-1.02907,-1.042810,-1.014931
18,vanilla G402S - vanilla WT,8.87539,8.80234,-0.07305,-0.079091,-0.067230
24,G402S masked - G402S vanilla,8.80234,8.70020,-0.10214,-0.112062,-0.090127
27,vanilla G402S - vanilla WT,10.23084,10.22924,-0.00160,-0.008610,0.005420
33,G402S masked - G402S vanilla,10.22924,10.12451,-0.10473,-0.115087,-0.095017
36,vanilla G402S - vanilla WT,7.56389,7.55164,-0.01225,-0.023523,-0.000160
42,G402S masked - G402S vanilla,7.55164,7.30954,-0.24210,-0.255308,-0.229495


,pdb_file,seed,model_number,recycle_number,source_column,position402_partner_CA_distance_A,partner,normalized_metric,sequence,protocol,mask_id,metric_definition
0,cav12_wt_short_unrelaxed_rank_001_alphafold2_p...,159,3,1,CA_GLY402_CA-PHE1519_CA,12.38,PHE1519,CA_POSITION402_CA-PHE1519_CA,WT,vanilla,unmasked,position-402 C-alpha to partner C-alpha; ident...
1,cav12_wt_short_unrelaxed_rank_001_alphafold2_p...,159,3,10,CA_GLY402_CA-PHE1519_CA,12.49,PHE1519,CA_POSITION402_CA-PHE1519_CA,WT,vanilla,unmasked,position-402 C-alpha to partner C-alpha; ident...
2,cav12_wt_short_unrelaxed_rank_001_alphafold2_p...,159,3,2,CA_GLY402_CA-PHE1519_CA,12.53,PHE1519,CA_POSITION402_CA-PHE1519_CA,WT,vanilla,unmasked,position-402 C-alpha to partner C-alpha; ident...
3,cav12_wt_short_unrelaxed_rank_001_alphafold2_p...,159,3,3,CA_GLY402_CA-PHE1519_CA,12.37,PHE1519,CA_POSITION402_CA-PHE1519_CA,WT,vanilla,unmasked,position-402 C-alpha to partner C-alpha; ident...
4,cav12_wt_short_unrelaxed_rank_001_alphafold2_p...,159,3,4,CA_GLY402_CA-PHE1519_CA,12.38,PHE1519,CA_POSITION402_CA-PHE1519_CA,WT,vanilla,unmasked,position-402 C-alpha to partner C-alpha; ident...
...,...,...,...,...,...,...,...,...,...,...,...,...
339740,cav12_g402s_short_masked_v3_50pct_unrelaxed_ra...,114,5,5,CA_SER402_CA-SER1535_CA,19.18,SER1535,CA_POSITION402_CA-SER1535_CA,G402S,masked,condition_specific_primary_mask,position-402 C-alpha to partner C-alpha; ident...
339741,cav12_g402s_short_masked_v3_50pct_unrelaxed_ra...,114,5,6,CA_SER402_CA-SER1535_CA,20.12,SER1535,CA_POSITION402_CA-SER1535_CA,G402S,masked,condition_specific_primary_mask,position-402 C-alpha to partner C-alpha; ident...
339742,cav12_g402s_short_masked_v3_50pct_unrelaxed_ra...,114,5,7,CA_SER402_CA-SER1535_CA,19.55,SER1535,CA_POSITION402_CA-SER1535_CA,G402S,masked,condition_specific_primary_mask,position-402 C-alpha to partner C-alpha; ident...
339743,cav12_g402s_short_masked_v3_50pct_unrelaxed_ra...,114,5,8,CA_SER402_CA-SER1535_CA,19.12,SER1535,CA_POSITION402_CA-SER1535_CA,G402S,masked,condition_specific_primary_mask,position-402 C-alpha to partner C-alpha; ident...


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)
